# Dispatch from a notebook

`dispatch.notebook` wraps the `dispatch job` CLI so Jobs can be launched and
monitored from a kernel on the Edge Node.

Locally, start the kernel from a shell that has sourced `mocks/dev-env.sh` so
the fake `impala-shell`, `klist`, and SMTP catcher are in place.

In [ ]:
from pathlib import Path

from dispatch.notebook import Dispatch, UsageError

workdir = Path.cwd()
Path(workdir / "demo_query.sql").write_text("SELECT 1 AS answer\n", encoding="utf-8")

d = Dispatch(cwd=workdir)
print("workspace:", d.workspace)
d

## Launch from a SQL file

Calling `launch()` is the confirmation the CLI spells `--yes`. The Job is
handed to a detached runner, so the kernel is free immediately.

In [ ]:
job = d.launch(
    source="SqlFile",
    destination="Csv",
    sql="demo_query.sql",
    table="demo_report",
)
job

## Monitor

`watch()` refreshes state and the log tail in one output cell until the Job
reaches a terminal state. Interrupting the kernel stops watching, not the Job.

In [ ]:
job.watch(poll_interval=1.0)

In [ ]:
print(job.state, job.exit_code, job.elapsed_seconds)
print(job.csv_path)
print(Path(job.csv_path).read_text(encoding="utf-8"))

## Supervise everything

`d.jobs()` reconciles stale manifests first, exactly like `dispatch job list`.

In [ ]:
d.jobs()

In [ ]:
print(job.logs(lines=5))

## SQL written here, loaded into a DataFrame

`sql()` saves the text as Inline SQL in the Notebook workspace and launches it
as an ordinary `SqlFile` Job, so the Result never lands beside your `.sql`
files. `to_df()` waits for the Job and reads the Result it exported.

In [ ]:
inline = d.sql("SELECT 1 AS answer, 'inline' AS origin")
frame = inline.to_df(poll_interval=1.0)
print("result:", inline.result_path)
frame

In [ ]:
print("dtypes:", dict(frame.dtypes.astype(str)))
print("columns:", inline.columns)
print("rows:", inline.rows())

# A bounded read of a table already in Impala, without writing SQL.
peek = d.table("aa_enc.events_existing", limit=5)
peek.to_df(poll_interval=1.0)

In [ ]:
# Keep a Result by name, then reclaim the workspace.
print("copied to:", inline.to_csv(workdir / "inline_result.csv"))
print(d.cleanup(older_than_days=0))

## Refusals raise

The Advisor gate is unchanged: SQL with error-severity findings is refused
until it is acknowledged explicitly.

In [ ]:
Path(workdir / "cross_join.sql").write_text(
    "SELECT a.x FROM aa_enc.t1 a CROSS JOIN aa_enc.t2 b\n", encoding="utf-8"
)

try:
    d.launch(source="SqlFile", destination="Csv", sql="cross_join.sql", table="crossed")
except UsageError as exc:
    print(f"refused (exit {exc.exit_code}): {exc}")

acknowledged = d.launch(
    source="SqlFile",
    destination="Csv",
    sql="cross_join.sql",
    table="crossed",
    acknowledge_advisor=True,
)
acknowledged.wait(poll_interval=1.0)